In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nextgen.db')

# Week 3 - SQL Queries: NextGenLearners Program Performance

In [3]:
pd.read_sql("SELECT * FROM applicants LIMIT 5;", conn)

,applicant_id,name,domain,university,application_date,status
0,1,Matthew Jones,Graphic Design,FAST NUCES,2026-08-01,Rejected
1,2,Anthony Romero,App Development,NED University,2026-08-01,Under Review
2,3,Benjamin Davis,App Development,Karachi University,2026-08-01,Rejected
3,4,Becky Walker,Data Science,UET Lahore,2026-08-01,Selected
4,5,Laura Sanders,Web Development,Comsats,2026-08-01,Selected


In [4]:
pd.read_sql("SELECT * FROM interns LIMIT 5;", conn)

,intern_id,applicant_id,domain,start_date,completion_status
0,1,4,Data Science,2026-08-16,Dropped Out
1,2,5,Web Development,2026-08-13,Completed
2,3,6,App Development,2026-08-12,Completed
3,4,10,App Development,2026-08-18,Completed
4,5,12,Data Science,2026-08-12,Completed


In [5]:
pd.read_sql("SELECT * FROM hackathon_scores LIMIT 5;", conn)

,intern_id,score,domain
0,2,81.0,Web Development
1,3,93.4,App Development
2,4,56.0,App Development
3,5,82.0,Data Science
4,6,64.2,Digital Marketing


## Query 1: How many interns completed each domain's program?

In [6]:
query1 = """
-- Counts completed interns grouped by domain
SELECT domain, COUNT(*) AS completed_count
FROM interns
WHERE completion_status = 'Completed'
GROUP BY domain
ORDER BY completed_count DESC;
"""
pd.read_sql(query1, conn)

,domain,completed_count
0,Digital Marketing,12
1,Web Development,10
2,App Development,10
3,Graphic Design,7
4,Data Science,3


**Takeaway:** [Domain] had the most completed interns ([number]), while
[Domain] had the fewest ([number]).

**Takeaway:** Digital Marketing had the most completed interns (12), followed
closely by Web Development and App Development (10 each). Data Science had
notably fewer completions (3), despite receiving a healthy number of
applications — worth investigating why.

## Query 2: What is the average hackathon score per domain?

In [7]:
query2 = """
-- Calculates the average hackathon score for each domain
SELECT domain, ROUND(AVG(score), 2) AS avg_score
FROM hackathon_scores
GROUP BY domain
ORDER BY avg_score DESC;
"""
pd.read_sql(query2, conn)

,domain,avg_score
0,Data Science,81.00
1,Graphic Design,79.00
2,App Development,72.66
3,Digital Marketing,71.94
4,Web Development,68.84


**Takeaway:** Despite having the fewest completed interns (3), Data Science
has the highest average hackathon score (81.00), followed by Graphic Design
(79.00). Web Development has the lowest average score (68.84) despite having
10 completions — suggesting quality may not always follow quantity across
domains.

## Query 3: Which interns scored above a threshold (top performers)?

In [8]:
query3 = """
-- Lists interns who scored 85 or above, as candidates for showcase/certificates
SELECT i.intern_id, i.domain, h.score
FROM interns i
JOIN hackathon_scores h ON i.intern_id = h.intern_id
WHERE h.score >= 85
ORDER BY h.score DESC;
"""
pd.read_sql(query3, conn)

,intern_id,domain,score
0,3,App Development,93.4
1,42,Graphic Design,88.8
2,51,Digital Marketing,86.6
3,9,Graphic Design,86.1


**Takeaway:** 4 interns scored 85 or above and qualify as top performers for
showcase/certificates. Intern #3 (App Development) had the highest score at
93.4, followed by two Graphic Design interns (#42 at 88.8, #9 at 86.1) and
one Digital Marketing intern (#51 at 86.6).

## Query 4: Conversion rate from "applied" to "completed" per domain

In [9]:
query4 = """
-- Compares total applicants vs completed interns per domain to find conversion rate
SELECT
    a.domain,
    COUNT(DISTINCT a.applicant_id) AS total_applicants,
    COUNT(DISTINCT i.intern_id) AS total_completed,
    ROUND(
        100.0 * COUNT(DISTINCT i.intern_id) / COUNT(DISTINCT a.applicant_id), 2
    ) AS conversion_rate_pct
FROM applicants a
LEFT JOIN interns i
    ON a.applicant_id = i.applicant_id AND i.completion_status = 'Completed'
GROUP BY a.domain
ORDER BY conversion_rate_pct DESC;
"""
pd.read_sql(query4, conn)

,domain,total_applicants,total_completed,conversion_rate_pct
0,App Development,26,10,38.46
1,Digital Marketing,34,12,35.29
2,Web Development,29,10,34.48
3,Graphic Design,33,7,21.21
4,Data Science,28,3,10.71


**Takeaway:** App Development has the highest applicant-to-completion
conversion rate (38.46%), followed closely by Digital Marketing (35.29%) and
Web Development (34.48%). Data Science stands out with a notably low
conversion rate (10.71%) despite 28 applicants — only 3 actually completed
the program, suggesting a bottleneck somewhere between selection and
completion for this domain that's worth investigating.

## Query 5 (My Choice): How many interns dropped out vs completed, per domain?

In [10]:
query5 = """
-- Compares completed vs dropped-out interns per domain to spot retention issues
SELECT
    domain,
    completion_status,
    COUNT(*) AS count
FROM interns
GROUP BY domain, completion_status
ORDER BY domain, completion_status;
"""
pd.read_sql(query5, conn)

,domain,completion_status,count
0,App Development,Completed,10
1,App Development,Dropped Out,2
2,Data Science,Completed,3
3,Data Science,Dropped Out,1
4,Digital Marketing,Completed,12
5,Digital Marketing,Dropped Out,7
6,Graphic Design,Completed,7
7,Graphic Design,Dropped Out,2
8,Web Development,Completed,10
9,Web Development,Dropped Out,1


**Why this question:** After seeing Data Science's low conversion rate in
Query 4, I wanted to check whether the drop-off happens because interns
start but don't finish (dropout), which would point to a program/engagement
issue rather than a selection issue.

**Takeaway:** Digital Marketing has the highest dropout count (7 out of 19
who started, ~37%), suggesting an engagement or difficulty issue during the
program itself. In contrast, Data Science only had 4 interns start in total
(3 completed, 1 dropped) out of 28 applicants — meaning its low conversion
rate from Query 4 is mainly driven by very few applicants getting selected
in the first place, not by people dropping out once they start.